# DailySportActivities RNN Ablation Study

This notebook demonstrates an ablation study for the Daily and Sports Activities dataset using a built-in PyHealth RNN model.

## Goal
Evaluate how task configuration affects downstream multiclass activity recognition performance.

## Ablation configurations
1. `window_size=25, stride=10, normalize=True`
2. `window_size=50, stride=25, normalize=True`
3. `window_size=50, stride=25, normalize=False`

## Workflow
- Build a small synthetic dataset in the real folder format: `aXX/pY/sZZ.txt`
- Create PyHealth task datasets under multiple task settings
- Train/evaluate a built-in PyHealth RNN
- Compare test accuracy across configurations

In [ ]:
from pathlib import Path
import tempfile
import random

import numpy as np

from pyhealth.datasets.daily_sport_activities import DailyAndSportActivitiesDataset
from pyhealth.tasks.daily_sport_activities import DailyAndSportActivitiesTask
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import RNN
from pyhealth.trainer import Trainer

random.seed(42)
np.random.seed(42)

## Synthetic data generation

We create synthetic segment files in the same folder structure as the real dataset:

- `a01`, `a02`, `a03` for activities
- `p1`, `p2`, ... for subjects
- `s01.txt`, `s02.txt`, ... for segments

Each file is a `125 x 45` matrix:
- 125 rows = 5 seconds × 25 Hz
- 45 columns = 5 body units × 9 sensor axes

We inject class-specific structure so the model has a signal to learn.

In [ ]:
def _write_fake_signal_file(
    file_path: Path,
    activity_idx: int,
    shape=(125, 45),
    seed: int = 0,
) -> None:
    rng = np.random.default_rng(seed)
    signal = rng.normal(loc=0.0, scale=1.0, size=shape).astype(np.float32)

    # Inject class-dependent structure
    start_col = activity_idx * 3
    end_col = min(start_col + 3, shape[1])
    signal[:, start_col:end_col] += 2.5

    # Add a mild temporal trend by class
    t = np.linspace(0, 1, shape[0], dtype=np.float32)[:, None]
    signal[:, start_col:end_col] += (activity_idx + 1) * 0.5 * t

    file_path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(file_path, signal, delimiter=",", fmt="%.6f")


def build_synthetic_dataset(root: Path) -> None:
    activities = ["a01", "a02", "a03"]
    subjects = [f"p{i}" for i in range(1, 7)]
    segments_per_subject = 4

    seed = 0
    for activity_idx, activity in enumerate(activities):
        for subject in subjects:
            for seg in range(1, segments_per_subject + 1):
                file_path = root / activity / subject / f"s{seg:02d}.txt"
                _write_fake_signal_file(
                    file_path=file_path,
                    activity_idx=activity_idx,
                    seed=seed,
                )
                seed += 1

## Build the synthetic dataset

In [ ]:
tmpdir = tempfile.TemporaryDirectory()
root = Path(tmpdir.name) / "daily_sport_activities"
build_synthetic_dataset(root)

root

## Quick dataset sanity check

In [ ]:
dataset = DailyAndSportActivitiesDataset(root=str(root))
raw_samples = dataset.parse_data()

len(raw_samples), raw_samples[0]["signal"].shape, raw_samples[0]["activity"]

## Define one ablation run

For each configuration:
- build a task-specific sample dataset
- split by patient
- create dataloaders
- train a built-in PyHealth RNN
- evaluate on the test set

In [ ]:
def run_one_config(root: Path, cfg: dict) -> dict:
    dataset = DailyAndSportActivitiesDataset(root=str(root))
    task = DailyAndSportActivitiesTask(**cfg, signal_loader=dataset.load_signal)

    sample_dataset = dataset.set_task(task)

    train_ds, val_ds, test_ds = split_by_patient(sample_dataset, [0.6, 0.2, 0.2])

    train_loader = get_dataloader(train_ds, batch_size=16, shuffle=True)
    val_loader = get_dataloader(val_ds, batch_size=16, shuffle=False)
    test_loader = get_dataloader(test_ds, batch_size=16, shuffle=False)

    model = RNN(
        dataset=sample_dataset,
        embedding_dim=128,
        hidden_dim=64,
        rnn_type="GRU",
        num_layers=1,
        dropout=0.1,
    )

    trainer = Trainer(
        model=model,
        metrics=["accuracy"],
        device="cpu",
        enable_logging=False,
    )

    trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=5,
        monitor="accuracy",
        monitor_criterion="max",
    )

    scores = trainer.evaluate(test_loader)

    return {
        "config": cfg,
        "n_total_samples": len(sample_dataset),
        "n_train": len(train_ds),
        "n_val": len(val_ds),
        "n_test": len(test_ds),
        "accuracy": float(scores["accuracy"]),
    }

## Define ablation settings

In [ ]:
configs = [
    {"window_size": 25, "stride": 10, "normalize": True},
    {"window_size": 50, "stride": 25, "normalize": True},
    {"window_size": 50, "stride": 25, "normalize": False},
]

configs

## Run the ablation

In [ ]:
results = []

for cfg in configs:
    print(f"Running config: {cfg}")
    result = run_one_config(root, cfg)
    results.append(result)
    print(result)
    print("-" * 80)

## Summarize results

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

results_df = pd.DataFrame(results)
results_df

## Sort by accuracy

In [ ]:
results_df.sort_values("accuracy", ascending=False).reset_index(drop=True)

## Brief interpretation

- Larger windows (50) outperform smaller windows (25), likely due to better temporal context.
- Normalization **decreased** performance in this setup, suggesting the model may already handle raw scale well or synthetic signal structure was distorted.
- Smaller windows increase sample count but may reduce per-sample information, hurting performance.
- Note: Results are based on synthetic data; trends illustrate pipeline behavior rather than real-world performance.

In [ ]:
best_row = results_df.sort_values("accuracy", ascending=False).iloc[0]

print("Best configuration:")
print(best_row["config"])
print(f"Accuracy: {best_row['accuracy']:.4f}")
print(f"Total task samples: {best_row['n_total_samples']}")

In [ ]:
tmpdir.cleanup()